# Generative AI as a Geography-Teaching Companion

*Geography + AI Teaching Notebooks · 3 of 3*

This notebook is a small **pattern library** for using generative AI in
geography teaching. The goal is not to hand thinking over to AI, but to
use it as a structured assistant — and to train students to read its
output critically.

Three reusable patterns are shown:

1. **Concept Tutor** — a constrained prompt that makes the model *teach*
   rather than merely answer.
2. **Map Reading** — a workflow for vision-capable models to extract
   structured information from a historical map.
3. **Field-Interview Pipeline** — speech-to-text plus structured
   summarisation for fieldwork notes.

Every cell runs with stubbed responses, so the notebook is reproducible
without API keys. Real API calls are shown but commented out.


In [ ]:
import json, textwrap
from dataclasses import dataclass

# A stubbed responder so the notebook runs without API access.
def call_llm(prompt: str, model: str = 'stub') -> str:
    if model == 'stub':
        return (
            'ONE-LINE DEFINITION: Transit time is the time water takes to'
            ' travel through a catchment from input to output.\n'
            'EXAMPLE: After a storm, rain landing on a hillslope may take'
            ' weeks to months to reach the stream as slow subsurface flow.\n'
            'FOLLOW-UP QUESTIONS: (1) How would a drought change transit'
            ' time? (2) Why can two catchments of equal size have very'
            ' different transit times?'
        )
    # Real usage (commented):
    # import openai
    # return openai.ChatCompletion.create(...)['choices'][0]['message']['content']
    return ''


## Pattern 1 · Concept Tutor

The system prompt forces the model to answer in three labelled parts: a
one-line definition, a concrete example, and two follow-up questions.
This turns a plain answer into a small teaching interaction.


In [ ]:
TUTOR_SYSTEM = textwrap.dedent('''\
    You are a geography teaching assistant.
    For every question, respond with exactly three labelled parts:
    (1) ONE-LINE DEFINITION
    (2) A CONCRETE EXAMPLE
    (3) TWO FOLLOW-UP QUESTIONS for the student to think about.
    Use plain prose. Do not exceed 120 words.''')

student_q = 'What is the transit time of water in a catchment?'
answer = call_llm(TUTOR_SYSTEM + '\n\nStudent: ' + student_q)
print(answer)


**In class.** The model output is *one* example answer. Students compare
it with the textbook, mark anything inaccurate, and judge whether the
example really fits. The grade rewards the critique, not the AI.


## Pattern 2 · Reading a historical map with a vision model

Vision-capable models can help transcribe old maps, but their output
must be checked. Forcing the answer into JSON makes it easy to compare
with the original and to load into a GIS.


In [ ]:
MAP_PROMPT = textwrap.dedent('''\
    You are reading a historical topographic map sheet. Return JSON with:
    - sheet_id (string)
    - place_names (list of {name, type})
    - waterways (list of {name, role})
    - uncertain_items (list of anything you could not read confidently)
    Be conservative: if unsure, put the item in uncertain_items.''')

# Real request (commented):
# response = openai.ChatCompletion.create(
#     model='gpt-4-vision', messages=[
#         {'role': 'system', 'content': MAP_PROMPT},
#         {'role': 'user', 'content': [{'type': 'image_url',
#                                       'image_url': 'file://map_sheet.jpg'}]}])

stub_response = {
    'sheet_id': 'historical_sheet_excerpt',
    'place_names': [
        {'name': 'Riverside Village', 'type': 'settlement'},
        {'name': 'Old Mill', 'type': 'landmark'},
    ],
    'waterways': [
        {'name': 'Main River', 'role': 'primary channel'},
        {'name': 'unnamed channel', 'role': 'possible former course'},
    ],
    'uncertain_items': ['a faint label near the map edge'],
}
print(json.dumps(stub_response, indent=2, ensure_ascii=False))


**In class.** Students compare the JSON with the original map. The
exercise builds three skills at once: map literacy, critical evaluation
of AI output, and structured-data thinking.


## Pattern 3 · A field-interview pipeline

During fieldwork, students often record short interviews with local
residents. The pipeline below converts speech to text and then asks an
LLM for a structured field summary — always staying close to the
speaker's own words.


In [ ]:
@dataclass
class InterviewSummary:
    interviewee_role: str
    key_themes: list[str]
    quoted_passages: list[str]
    open_questions: list[str]

STRUCTURED_PROMPT = textwrap.dedent('''\
    Read the interview transcript and produce JSON with: interviewee_role,
    key_themes (3-5), quoted_passages (verbatim), open_questions.
    Stay close to the speaker. Do not invent information.''')

sample_transcript = (
    'We have farmed this land for three generations. The irrigation water'
    ' used to arrive on the same date every year, but lately the schedule'
    ' shifts with the reservoir level, so in dry years we plant less.'
)
# response = call_llm(STRUCTURED_PROMPT + '\n\n' + sample_transcript)
stub = InterviewSummary(
    interviewee_role='Smallholder farmer',
    key_themes=['Irrigation scheduling', 'Reservoir-driven variability',
                'Multi-generational farming'],
    quoted_passages=['in dry years we plant less'],
    open_questions=['How is the irrigation schedule communicated to farmers today?'],
)
print(stub)


## Reflection: where AI fits in the curriculum

| Pattern | Role of the AI | Student learning target |
|---|---|---|
| Concept Tutor | Scaffold | Comparison and critique |
| Map Reading | Extractor | Map literacy + data thinking |
| Field-Interview Pipeline | Stenographer | Field method |

In every pattern, **the AI's output is a draft, not a verdict.** Students
are assessed on how they evaluate, correct, and extend that draft —
never on whether the AI happened to be right.

This pattern library is open and will grow as more classroom-tested
uses of AI in geography teaching are added.
